In [ ]:
from pathlib import Path
import sys, subprocess, tempfile, os, signal
from IPython.display import FileLink, display
working=Path('/kaggle/working')
receipt=working/'imc_ns64_dependency_receipt.zip'
if receipt.exists(): raise FileExistsError('Use a fresh session to preserve the previous receipt')
script=working/'phase3c1_verify.py'
script.write_text('"""Independent offline dependency checks. No benchmark images are read."""\nimport hashlib\nimport json\nimport os\nimport platform\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport zipfile\nfrom pathlib import Path\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with path.open(\'rb\') as f:\n        for chunk in iter(lambda:f.read(4*1024**2), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef probe(code, cwd, args=(), timeout=60):\n    start=time.monotonic()\n    measurement = "\\nimport sys\\nif sys.platform==\'linux\':\\n import resource,json; print(\'RESOURCE\',json.dumps(dict(cpu_peak_rss_bytes=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss*1024,gpu_peak_allocated_bytes=__import__(\'torch\').cuda.max_memory_allocated() if \'torch\' in __import__(\'sys\').modules and __import__(\'torch\').cuda.is_available() else 0)))"\n    try:\n        result = subprocess.run([sys.executable,\'-c\',code+measurement,*map(str,args)],cwd=cwd,\n                                capture_output=True,text=True,timeout=timeout)\n        return {\'status\':\'PASS\' if result.returncode==0 else \'BLOCKED\',\n                \'seconds\':time.monotonic()-start, \'details\':(result.stdout+result.stderr)[-3000:]}\n    except (OSError, subprocess.TimeoutExpired) as exc:\n        return {\'status\':\'BLOCKED\',\'details\':str(exc)}\n\n\ndef input_files(roots, filename):\n    for root in roots:\n        for directory, names, files in os.walk(root):\n            names[:] = [n for n in names if n not in (\'competitions\',\'train\',\'test\') and not n.startswith(\'image-matching-challenge-\')]\n            if filename in files:\n                yield Path(directory)/filename\n\n\ndef locate(roots, asset):\n    paths = sorted(set(input_files(roots,asset[\'filename\'])))\n    found = [{\'path\':str(p),\'sha256\':sha(p),\'bytes\':p.stat().st_size} for p in paths]\n    matches = [p for p in found if p[\'sha256\']==asset[\'sha256\']]\n    return found, Path(matches[0][\'path\']) if matches else None\n\n\ndef verify(manifest, roots, bundle, receipt):\n    os.environ.update(HF_HUB_OFFLINE=\'1\',TRANSFORMERS_OFFLINE=\'1\')\n    start=time.monotonic()\n    checks={}\n    record={\'platform\':platform.platform(),\'python\':sys.version,\'checks\':checks,\'assets\':{},\n            \'environment_freeze\':subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True)}\n    work=Path(\'/kaggle/working\')\n    record[\'build_evidence\']={}\n    for name in [\'ns64_wheelhouse/asmk_build_adaptation.json\',\'ns64_wheelhouse/SHA256SUMS.json\',\'ns64_toolchain_explicit.txt\']:\n        p=work/name\n        if p.is_file():record[\'build_evidence\'][name]=p.read_text()\n    record[\'installed_extension_origins\']={}\n    import importlib.metadata\n    for name in [\'asmk\',\'curope\']:\n        try:record[\'installed_extension_origins\'][name]=importlib.metadata.distribution(name).read_text(\'direct_url.json\')\n        except importlib.metadata.PackageNotFoundError:record[\'installed_extension_origins\'][name]=None\n    def save():\n        record[\'seconds\']=time.monotonic()-start\n        temp=receipt.with_suffix(\'.tmp\')\n        with zipfile.ZipFile(temp,\'w\',zipfile.ZIP_DEFLATED) as z:\n            z.writestr(\'result.json\',json.dumps(record,indent=2))\n            z.writestr(\'manifest.json\',json.dumps(manifest,indent=2))\n        temp.replace(receipt)\n    record[\'status\']=\'PARTIAL\'\n    save()\n    try:\n        with tempfile.TemporaryDirectory() as tmp:\n            source=Path(tmp)\n            if sha(bundle)!=manifest[\'archive_sha256\']:\n                raise ValueError(\'Source bundle SHA256 mismatch\')\n            with zipfile.ZipFile(bundle) as z:z.extractall(source)\n            if sha(source/manifest[\'config\'])!=manifest[\'config_sha256\']:\n                raise ValueError(\'Configuration SHA256 mismatch\')\n            if sys.version_info[:2] == (3,11) and sys.platform==\'linux\':\n                wheels=sorted((source/\'bundle/oss\').rglob(\'*.whl\'))\n                expected=manifest[\'bundled_wheels\']\n                if any(sha(w)!=expected[w.name] for w in wheels):\n                    raise ValueError(\'Source wheel integrity error\')\n                extra=next((p for p in input_files(roots,manifest[\'pycolmap_filename\']) if sha(p)==manifest[\'pycolmap_sha256\']), None)\n                if extra:wheels.append(extra)\n                target=source/\'offline_site\'\n                install=subprocess.run([sys.executable,\'-m\',\'pip\',\'install\',\'--no-index\',\'--no-deps\',\'--target\',str(target),*map(str,wheels)],capture_output=True,text=True,timeout=180)\n                checks[\'bundled_wheel_install\']={\'status\':\'PASS\' if install.returncode==0 else \'BLOCKED\',\'details\':install.stderr[-1600:]}\n                os.environ[\'PYTHONPATH\']=os.pathsep.join([str(target),os.environ.get(\'PYTHONPATH\',\'\')])\n            for name,code in manifest[\'probes\'].items():\n                checks[name]=probe(code,source)\n                save()\n            verified={}\n            for key,asset in manifest[\'assets\'].items():\n                found,path=locate(roots,asset)\n                if path and key==\'dinov2\':\n                    if any(not (path.parent/n).is_file() or sha(path.parent/n)!=v for n,v in asset[\'sidecars\'].items()):path=None\n                record[\'assets\'][key]={\'files\':found,\'status\':\'PASS\' if path else \'BLOCKED\'}\n                if path:verified[key]=path\n                save()\n            for key,code in manifest[\'loaders\'].items():\n                if key==\'codebook\':\n                    checks[\'load_codebook\']=dict(checks[\'load_retrieval\'])\n                    save()\n                    continue\n                required={\'retrieval\':[\'mast3r\',\'retrieval\',\'codebook\'],\'codebook\':[\'mast3r\',\'retrieval\',\'codebook\']}.get(key,[key])\n                if all(k in verified for k in required):\n                    paths=[verified[k] for k in required]\n                    if key in (\'retrieval\',\'codebook\') and paths[-1].parent!=paths[-2].parent:\n                        result={\'status\':\'BLOCKED\',\'details\':\'Retriever requires colocated head and codebook\'}\n                    else:result=probe(code,source,paths,120)\n                else:result={\'status\':\'BLOCKED\',\'details\':\'Missing verified prerequisite files\'}\n                checks[\'load_\'+key]=result\n                save()\n            record[\'status\']=\'PASS\' if all(x[\'status\']==\'PASS\' for x in list(checks.values())+list(record[\'assets\'].values())) else \'BLOCKED\'\n    except Exception as exc:\n        record.update(status=\'BLOCKED\',error=str(exc))\n    finally:save()\n    return record\n\nroot=Path(\'/kaggle/input\')\nbundle=next(input_files([root],\'ns64_pilot_source.zip.bin\'),root/\'missing-source\')\nr=verify({\'revision\': \'07b5695b99a8ae4725d5d952f0065c2157fadd7d\', \'archive_sha256\': \'c6035c596a484331ebfaf8d3b6368b4a32acb00b3bebe670a8215c78b6c12549\', \'config\': \'conf/pipeline/imc2025/mast3rhybrid/mast3rhybrid-022-c.yaml\', \'config_sha256\': \'e7829be60865192d3ddcc8555b94bc8678415e265a566d8ce8386ef17b0cc763\', \'assets\': {\'superpoint\': {\'filename\': \'superpoint_v1.pth\', \'alias\': \'MAGICLEAP_SUPERPOINT\', \'url\': \'https://raw.githubusercontent.com/magicleap/SuperGluePretrainedNetwork/ddcf11f42e7e0732a0c4607648f9448ea8d73590/models/weights/superpoint_v1.pth\', \'sha256\': \'52b6708629640ca883673b5d5c097c4ddad37d8048b33f09c8ca0d69db12c40e\', \'size\': 5206086, \'format\': \'SuperPoint CNN, state dict\', \'module\': \'models._research_only.magicleap.superpoint\', \'license\': \'Magic Leap academic or nonprofit noncommercial research license\', \'published_sha256\': None, \'observed_in_previous_kaggle\': False, \'load_status\': \'UNTESTED\'}, \'aliked\': {\'filename\': \'aliked-n16.pth\', \'alias\': \'ALIKED_LIGHTGLUE_N16\', \'url\': \'https://raw.githubusercontent.com/Shiaoming/ALIKED/main/models/aliked-n16.pth\', \'sha256\': \'5be8704840ed662d9d8c561bf7279c222092674e7eb05fd0feab94899e9d82f2\', \'size\': 2738091, \'format\': \'ALIKED N16 state dict\', \'module\': \'features.lightglue.aliked\', \'license\': \'ALIKED BSD-3-Clause; retain upstream notices\', \'published_sha256\': None, \'observed_in_previous_kaggle\': True, \'load_status\': \'UNTESTED\'}, \'mast3r\': {\'filename\': \'MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth\', \'alias\': \'MAST3R\', \'url\': \'https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth\', \'sha256\': \'e28f91b488554653e2b46ddae9c78c1143e0bcb2e27d3e26cdb0b717f1568eb2\', \'size\': 2754910614, \'format\': \'AsymmetricMASt3R ViT-Large / base decoder 512 PyTorch checkpoint\', \'module\': \'models.mast3r.model\', \'license\': \'CC BY-NC-SA 4.0 and upstream training dataset conditions, including MapFree. Review CHECKPOINTS_NOTICE.\', \'published_sha256\': None, \'observed_in_previous_kaggle\': False, \'load_status\': \'UNTESTED\'}, \'retrieval\': {\'filename\': \'MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric_retrieval_trainingfree.pth\', \'alias\': \'MAST3R_RETRIEVAL\', \'url\': \'https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric_retrieval_trainingfree.pth\', \'sha256\': \'000a324f06f9369ab3e32d03dd4b8a838d48e98e3f3bf9d7dd034dce1bfd4c50\', \'size\': 8399111, \'format\': \'training-free retrieval head checkpoint\', \'module\': \'mast3r.retrieval.processor\', \'license\': \'CC BY-NC-SA 4.0 and upstream training dataset conditions, including MapFree. Review CHECKPOINTS_NOTICE.\', \'published_sha256\': None, \'observed_in_previous_kaggle\': False, \'load_status\': \'UNTESTED\'}, \'codebook\': {\'filename\': \'MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric_retrieval_codebook.pkl\', \'alias\': \'MAST3R_RETRIEVAL_CODEBOOK\', \'url\': \'https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric_retrieval_codebook.pkl\', \'sha256\': \'700af9e31ad0ca6159955dc80b4fd6e14234780b52f5c91884f6553991933c1a\', \'size\': 268435691, \'format\': \'64k ASMK codebook pickle\', \'module\': \'asmk.asmk_method\', \'license\': \'CC BY-NC-SA 4.0 and upstream training dataset conditions, including MapFree. Review CHECKPOINTS_NOTICE.\', \'published_sha256\': None, \'observed_in_previous_kaggle\': False, \'load_status\': \'UNTESTED\'}, \'isc\': {\'filename\': \'isc_ft_v107.pth.tar\', \'alias\': \'ISC\', \'url\': \'https://github.com/lyakaap/ISC21-Descriptor-Track-1st/releases/download/v1.0.1/isc_ft_v107.pth.tar\', \'sha256\': \'512390a15ab52f8ab5adc14a315b157a5d6e732e3361c7a0dce5be6e95a86d30\', \'size\': 420418996, \'format\': \'ISCNet EfficientNetV2-M fine-tuned v107 PyTorch tar checkpoint\', \'module\': \'models.isc.model\', \'license\': \'ISC repository MIT; training-data rights and checkpoint redistribution permission require separate review\', \'published_sha256\': None, \'observed_in_previous_kaggle\': False, \'load_status\': \'UNTESTED\'}, \'dinov2\': {\'filename\': \'pytorch_model.bin\', \'alias\': \'DINOV2_BASE\', \'url\': \'https://huggingface.co/facebook/dinov2-base/blob/main/pytorch_model.bin\', \'sha256\': \'014965d9e330e7f4bff8ddcbee9df5e4f2ca032b2f5180942a6edb454783e75d\', \'size\': 346392713, \'sidecars\': {\'config.json\': \'f7ff4cfa73d2f70647dbf6950541ad25d73082d54c2e7e9bded160c7656b2a70\', \'preprocessor_config.json\': \'14e780d86fa1861f8751f868d7f45425b5feb55c38ca26f152ca5097ab30f828\'}, \'format\': \'ViT-B/14 Transformers directory\', \'module\': \'global_descriptors.dinov2\', \'license\': \'Meta Apache-2.0 for code and weights\', \'published_sha256\': \'014965d9e330e7f4bff8ddcbee9df5e4f2ca032b2f5180942a6edb454783e75d\', \'observed_in_previous_kaggle\': True, \'load_status\': \'UNTESTED\'}}, \'probes\': {\'python\': "import platform,sys; assert platform.system()==\'Linux\' and sys.version_info[:2]==(3,11);print(sys.version)", \'gpu\': "import subprocess;subprocess.run([\'nvidia-smi\',\'--query-gpu=name,driver_version,memory.total\',\'--format=csv\'],check=True)", \'torch\': "import torch;print(torch.__version__,torch.version.cuda,torch.cuda.get_device_capability());assert torch.__version__.split(\'+\')[0]==\'2.5.1\';print(torch.ones(1,device=\'cuda\').item())", \'distributed\': \'import torch.distributed as d;from distributed import DistConfig;print(DistConfig.single());assert d.is_nccl_available()\', \'pycolmap\': "import pycolmap as p;print(p.__version__);assert p.__version__==\'3.11.1\';p.IncrementalPipelineOptions();p.SiftMatchingOptions();assert callable(p.incremental_mapping) and callable(p.match_exhaustive)", \'driver_toolkit\': "import shutil,subprocess,torch;print(torch.version.cuda);assert torch.version.cuda==\'12.1\';print(subprocess.check_output([\'nvidia-smi\',\'--query-gpu=driver_version\',\'--format=csv,noheader\'],text=True));print(shutil.which(\'nvcc\') or \'runtime only\')", \'pipeline\': "from kernel import run;from config import PipelineConfig;print(PipelineConfig.load_config(\'conf/pipeline/imc2025/mast3rhybrid/mast3rhybrid-022-c.yaml\').type)", \'asmk.hamming\': \'import torch;import asmk.hamming\', \'asmk.asmk_method\': \'import torch;import asmk.asmk_method\', \'curope\': \'import torch;import curope\', \'mast3r.model\': \'import torch;import mast3r.model\', \'dust3r.inference\': \'import torch;import dust3r.inference\', \'lightglue\': \'import torch;import lightglue\', \'models._research_only.magicleap.superpoint\': \'import torch;import models._research_only.magicleap.superpoint\', \'models.isc.model\': \'import torch;import models.isc.model\', \'faiss\': \'import torch;import faiss\', \'torchvision\': "import torchvision;assert torchvision.__version__.split(\'+\')[0]==\'0.20.1\'", \'numpy\': "import numpy;assert numpy.__version__.split(\'+\')[0]==\'1.26.4\'", \'kornia\': "import kornia;assert kornia.__version__.split(\'+\')[0]==\'0.8.0\'", \'transformers\': "import transformers;assert transformers.__version__.split(\'+\')[0]==\'4.47.0\'"}, \'loaders\': {\'superpoint\': \'import sys,torch;from pathlib import Path;p=Path(sys.argv[1]);from models._research_only.magicleap.superpoint import SuperPoint;SuperPoint(dict(model_path=str(p),fix_sampling=True,max_keypoints=4096,keypoint_threshold=.0005)).eval().cuda()\', \'aliked\': "import sys,torch;from pathlib import Path;p=Path(sys.argv[1]);from features.lightglue.aliked import ALIKED;ALIKED(str(p),model_name=\'aliked-n16\',max_num_keypoints=4096).eval().cuda()", \'dinov2\': \'import sys,torch;from pathlib import Path;p=Path(sys.argv[1]);from transformers import AutoModel,AutoImageProcessor;AutoModel.from_pretrained(p.parent,local_files_only=True).eval().cuda();AutoImageProcessor.from_pretrained(p.parent,local_files_only=True)\', \'mast3r\': "import sys,torch;from pathlib import Path;p=Path(sys.argv[1]);from models.mast3r.model import get_mast3r_model;m=get_mast3r_model(p,torch.device(\'cuda\'));v={\'instance\':[\'left\'],\'img\':torch.zeros(1,3,224,224,device=\'cuda\'),\'true_shape\':torch.tensor([[224,224]])};\\nwith torch.inference_mode(): a,b=m(v,dict(v,instance=[\'right\']));assert torch.isfinite(a[\'pts3d\']).all();torch.cuda.synchronize()", \'retrieval\': "import sys,torch;from pathlib import Path;p=Path(sys.argv[1]);from models.mast3r.model import get_mast3r_model;from mast3r.retrieval.processor import Retriever; m=get_mast3r_model(p,torch.device(\'cuda\'));r=Retriever(sys.argv[2],backbone=m,device=\'cuda\');assert r.asmk is not None", \'codebook\': "import sys,torch;from pathlib import Path;p=Path(sys.argv[1]);from models.mast3r.model import get_mast3r_model;from mast3r.retrieval.processor import Retriever; m=get_mast3r_model(p,torch.device(\'cuda\'));r=Retriever(sys.argv[2],backbone=m,device=\'cuda\');assert r.asmk is not None", \'isc\': "import sys,torch;from pathlib import Path;p=Path(sys.argv[1]);from models.isc.model import create_model;create_model(p,torch.device(\'cuda\'))"}, \'bundled_wheels\': {\'croco-0.1.2-py3-none-any.whl\': \'ee26071c4c63f454d043c38f851cc692ee9d956bcd18a90996b21325df7e23a2\', \'dad-0.2.0-py3-none-any.whl\': \'d45d0f363cd3c793ca98721b689342d4ae3344eac6667a26137110b5eb5c27f8\', \'dedode-0.0.1-py3-none-any.whl\': \'436f876a5b57957f660ab652f019ff85379057a39ade5f16396124d82474c142\', \'depth_anything_v2-0.0.1-py3-none-any.whl\': \'c19b2c9d4b18d45d758757f6c215087b5edb08f3641438915db84e6d6672594f\', \'dust3r-0.1.0-py3-none-any.whl\': \'b4313794fd89ff3d7ee7ebdf1210ed852f6f505bbffad9fdfe712fabe4814b2b\', \'hloc-1.5-py3-none-any.whl\': \'44a53f7e6ea41915dd6c48d9dce20a393fd2186ad109709c5003e62a84e49939\', \'lightglue-0.0-py3-none-any.whl\': \'56383e98ab14c77b2f2c8214c43b2ee33b8f11bf622c4b6e8b5b335ce421adfd\', \'mast3r-0.1.0-py3-none-any.whl\': \'6a5367dd9d097d6549129a74ab15b29daf9f397fd8b04242627b79b53679f0d8\', \'pyrender-0.1.45-py3-none-any.whl\': \'619169d5468873900b414af58f89d3a6bd61d411e877810f3ff9291a4db4063b\', \'vggt-0.0.1-py3-none-any.whl\': \'2f25a6f22fda1b6f0c64f0dfbe795075e4f0e043eec3322fe1a1d6cf1a47f93a\', \'moge-1.0.0-py3-none-any.whl\': \'9940118078e5c034b91e2d0f27982ff6e00a914e7c0d967c6db881c6d62fd3a3\', \'utils3d-0.0.2-py3-none-any.whl\': \'e6674c15a840e54adc063f05e9816edf0ec533fc112dba0d986be77edd14d117\', \'mpsfm-0.1.0-py3-none-any.whl\': \'7c28365aec2bea6d1127296a85cfbfd69820cb45102200b90e18ca866accebfe\', \'dsine-0.0.0-py3-none-any.whl\': \'3ff645d126f5c081e9077252a65d28cdc4aad38201ba14f65ddb986ae32e6f43\', \'mono-0.0.0-py3-none-any.whl\': \'479f3fc45d9638a14653080f1bb1af546303402b3b8e7f5aae54e6dd65271aa3\', \'superglue_pretrained_network-0.0.0-py3-none-any.whl\': \'fd0f5aca1dcfc3c7c744fb40e8c03965089a793d955ee19ff8f529e444dbfd14\'}, \'pycolmap_filename\': \'pycolmap-3.11.1-cp311-cp311-manylinux_2_28_x86_64.whl\', \'pycolmap_sha256\': \'9bb98585e540a403df718b26ffdf475eba74891d16bf59ba94eef0f3acb827a5\'},[root,Path(\'/kaggle/working/ns64_weights\')],bundle,Path(\'/kaggle/working/imc_ns64_dependency_receipt.zip\'))\nprint(r[\'status\'],{k:v[\'status\'] for k,v in r[\'checks\'].items()})\n')
try:
    child_env=os.environ.copy()
    child_env['MPLBACKEND']='Agg'
    child=subprocess.Popen(['/kaggle/working/ns64_env/bin/python',str(script)],env=child_env,start_new_session=True)
    try:
        if child.wait(timeout=1800): raise RuntimeError('Dependency worker failed')
    except subprocess.TimeoutExpired:
        os.killpg(child.pid, signal.SIGKILL)
        child.wait()
        print('Time budget reached. Download partial receipt.')
finally:
    if receipt.exists(): display(FileLink(str(receipt)))
